# 01. QC & トリミング        RNA-seq FASTQデータの品質管理とアダプター除去を行います。**カーネル**: RNA-seq (Python) / rnaseq_env  **必要ツール**: FastQC, MultiQC, Trim Galore  **入力**: `raw_data/` 内のFASTQファイル (Paired-end)  **出力**: `trimmed/` にトリミング済みFASTQ, `qc_reports/` にQCレポート

## 設定

In [ ]:
import os, glob, subprocess, time, logging

# === 設定 ===
RAW_DATA_DIR = "raw_data"
TRIMMED_DIR = "trimmed"
QC_DIR = "qc_reports"
QC_PRETRIM_DIR = os.path.join(QC_DIR, "pretrim")
QC_POSTTRIM_DIR = os.path.join(QC_DIR, "posttrim")
THREADS = 8

# トリミングパラメータ
TRIM_QUALITY = 20       # 品質スコア閾値
TRIM_MIN_LENGTH = 36    # 最小リード長

# ディレクトリ作成
for d in [TRIMMED_DIR, QC_PRETRIM_DIR, QC_POSTTRIM_DIR]:
    os.makedirs(d, exist_ok=True)

# ロギング設定
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(LOG_DIR, "01_qc_and_trimming.log")),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

def run_cmd(cmd, step_name=""):
    """外部コマンドを実行し、失敗時にエラーを投げる"""
    logger.info(f"実行: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        logger.error(f"{step_name} 失敗 (exit code {result.returncode})")
        logger.error(f"stderr: {result.stderr}")
        raise RuntimeError(f"{step_name} が失敗しました:\n{result.stderr}")
    logger.info(f"{step_name} 完了")
    return result

print("設定完了")

## ツール確認

In [ ]:
# 必要ツールの存在確認
missing_tools = []
for tool in ["fastqc", "multiqc", "trim_galore"]:
    ret = subprocess.run(f"which {tool}", shell=True, capture_output=True)
    status = "OK" if ret.returncode == 0 else "NOT FOUND"
    print(f"  {tool}: {status}")
    if ret.returncode != 0:
        missing_tools.append(tool)

if missing_tools:
    raise EnvironmentError(f"必要なツールが見つかりません: {', '.join(missing_tools)}")

## FASTQファイルの検出

In [ ]:
# Paired-end FASTQファイルを検出
r1_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "*_R1*.fastq.gz")))
r2_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "*_R2*.fastq.gz")))

print(f"検出されたペア数: {{len(r1_files)}}")
for r1, r2 in zip(r1_files, r2_files):
    print(f"  R1: {{os.path.basename(r1)}}")
    print(f"  R2: {{os.path.basename(r2)}}")
    print()

assert len(r1_files) == len(r2_files), "R1とR2のファイル数が一致しません！"
assert len(r1_files) > 0, "FASTQファイルが見つかりません。raw_data/ を確認してください。"

## サンプルメタデータの確認

In [ ]:
import pandas as pd

metadata = pd.read_csv("sample_metadata.csv")

# 条件名のバリデーション
for cond in metadata["condition"].unique():
    if " " in str(cond):
        raise ValueError(
            f"条件名にスペースが含まれています: '{cond}'\n"
            f"スペースをアンダースコア(_)に置換してください。\n"
            f"例: 'compound A 0.5 nM' → 'CompA_0.5nM'"
        )

print("サンプルメタデータ:")
print(metadata.to_string(index=False))
print(f"\n条件: {metadata['condition'].unique().tolist()}")
print(f"条件数: {metadata['condition'].nunique()}, サンプル数: {len(metadata)}")

## Step 1: トリミング前 QC (FastQC)

In [ ]:
%%time
all_fastqs = " ".join(r1_files + r2_files)
cmd = f"fastqc -t {THREADS} -o {QC_PRETRIM_DIR} {all_fastqs}"
run_cmd(cmd, "FastQC (トリミング前)")
print("FastQC (トリミング前) 完了")

## Step 2: トリミング (Trim Galore)

In [ ]:
%%time
for r1, r2 in zip(r1_files, r2_files):
    cmd = (
        f"trim_galore --paired --quality {TRIM_QUALITY} --length {TRIM_MIN_LENGTH} "
        f"--cores {THREADS} "
        f"-o {TRIMMED_DIR} "
        f"{r1} {r2}"
    )
    print(f"処理中: {os.path.basename(r1)}")
    run_cmd(cmd, f"Trim Galore ({os.path.basename(r1)})")

print("\nTrim Galore 完了")
print(f"トリミング済みファイル: {TRIMMED_DIR}/")

## Step 3: トリミング後 QC (FastQC)

In [ ]:
%%time
trimmed_fastqs = " ".join(sorted(glob.glob(os.path.join(TRIMMED_DIR, "*_val_*.fq.gz"))))
if not trimmed_fastqs:
    raise FileNotFoundError(f"トリミング済みファイルが見つかりません: {TRIMMED_DIR}/*_val_*.fq.gz")
cmd = f"fastqc -t {THREADS} -o {QC_POSTTRIM_DIR} {trimmed_fastqs}"
run_cmd(cmd, "FastQC (トリミング後)")
print("FastQC (トリミング後) 完了")

## Step 4: MultiQC レポート統合

In [ ]:
%%time
cmd = f"multiqc {QC_DIR} -o {QC_DIR} -f --title 'RNA-seq QC Report'"
run_cmd(cmd, "MultiQC")
print(f"MultiQCレポート: {QC_DIR}/multiqc_report.html")

## サマリー- トリミング前後のQCレポートを `qc_reports/multiqc_report.html` で確認してください- 品質に問題がなければ次のノートブック `02_Mapping_and_Counting.ipynb` に進みます### チェックポイント- Per base quality scores が Q20 以上であること- Adapter content が除去されていること  - 極端にリード数が少ないサンプルがないこと